# Azure OpenAI RAG with Vector Store (Responses API)

This notebook demonstrates a simple retrieval-augmented generation (RAG) flow using:

- An existing **Azure OpenAI Vector Store** (already deployed)
- Your deployed **Azure OpenAI model deployment** (used by Responses API)

No Azure AI Search is used.

You will: (1) configure connection variables, (2) create a client, (3) call `responses.create` with `file_search` tool resources, and (4) print the answer.

## 1) Configuration (fill in your values)

Set these as environment variables (recommended) or edit the defaults below.

Required:
- `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_API_VERSION` (if not set, we default to `2025-03-01-preview`)
- `AZURE_OPENAI_DEPLOYMENT` (your deployed Azure OpenAI model name/deployment)
- `AZURE_OPENAI_VECTOR_STORE_ID` (the Vector Store you already deployed)

Optional:
- `AZURE_OPENAI_ASSISTANT_NAME` (only used for display)
- `LANGSMITH_API_KEY` (optional; set this to enable LangSmith tracing for API calls)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2025-03-01-preview')

AZURE_OPENAI_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT')
AZURE_OPENAI_VECTOR_STORE_ID = os.getenv('AZURE_OPENAI_VECTOR_STORE_ID')

missing = [
    name
    for name, value in [
        ('AZURE_OPENAI_API_KEY', AZURE_OPENAI_API_KEY),
        ('AZURE_OPENAI_ENDPOINT', AZURE_OPENAI_ENDPOINT),
        ('AZURE_OPENAI_DEPLOYMENT', AZURE_OPENAI_DEPLOYMENT),
        ('AZURE_OPENAI_VECTOR_STORE_ID', AZURE_OPENAI_VECTOR_STORE_ID),
    ]
    if not value
]

if missing:
    raise ValueError('Missing required env vars: ' + ', '.join(missing))

print('Config OK')
print('API version:', AZURE_OPENAI_API_VERSION)
print('Model deployment:', AZURE_OPENAI_DEPLOYMENT)
print('Vector store:', AZURE_OPENAI_VECTOR_STORE_ID)


## 2) Create an Azure OpenAI client

In [ ]:
import langsmith
from openai import AsyncAzureOpenAI

LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')
if LANGSMITH_API_KEY:
    os.environ['LANGSMITH_API_KEY'] = LANGSMITH_API_KEY

client = AsyncAzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)

# Wrap the Azure OpenAI client so Responses API calls are traced by LangSmith.
client = langsmith.wrappers.wrap_openai(client)

print('Client created with LangSmith tracing' if LANGSMITH_API_KEY else 'Client created')

## 3) Responses API call with vector store retrieval

In [ ]:
import asyncio
from IPython.display import Markdown, display
from langsmith import traceable

@traceable(name="Azure RAG and WebSearch tools")
async def run_rag_responses(question: str):
    response = await client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        input=question,
        tools=[
            {
                'type': 'file_search',
                'vector_store_ids': [AZURE_OPENAI_VECTOR_STORE_ID],
            },
            {
                'type': 'web_search',
            }
        ],
    )

    output_parts = []
    for item in response.output:
        if getattr(item, 'type', None) == 'message':
            for content in getattr(item, 'content', []) or []:
                if getattr(content, 'type', None) == 'output_text':
                    output_parts.append(getattr(content, 'text', ''))
    return '\n'.join([p for p in output_parts if p]).strip()

question = 'Explain GRPO for LLMs with an example.'
answer = await run_rag_responses(question)
display(Markdown(answer))
#print(answer)

GRPO (often written **Group Relative Policy Optimization**) is a **reinforcement-learning–style** training method for LLMs that uses a **group of candidate outputs** per prompt, then updates the model by increasing the probability of the candidates that score higher than the group average (a *relative* advantage), rather than relying on an absolute value target.

### Core idea
1. For each prompt, sample **K** different completions from the current policy (the LLM).
2. Score each completion with a **reward function** (could be a reward model, rule-based checks, verifier model, etc.).
3. Compute a **relative advantage** within the group, e.g.:
   - Advantage ≈ `reward_i − mean(reward_1..reward_K)`
4. Update the policy to **upweight** outputs with positive advantage and **downweight** those with negative advantage.
5. Typically include a **KL penalty** against a reference policy to avoid drifting too far from a base model.

### Why “group” and “relative” matter
- **Group**: you compare samples generated from the same prompt under the same model snapshot.
- **Relative**: you don’t need reward calibration across prompts—only *within* the group.

---

## Example (toy math)
Assume:
- You have a prompt:  
  “Compute the value of 2+2. Reply with just the number.”
- The model samples **K=4** candidates:
  1. “4”  
  2. “4.”  
  3. “5”  
  4. “The answer is 4.”

### Step 1: Score with a reward
Suppose your reward is a simple rule: **1 if the response is exactly the number 4, else 0**.
- Candidate rewards:  
  1) “4” → 1  
  2) “4.” → 0  
  3) “5” → 0  
  4) “The answer is 4.” → 0  

Rewards: `[1, 0, 0, 0]`

### Step 2: Compute the group baseline and advantages
- Mean reward = `(1+0+0+0)/4 = 0.25`
- Advantages (reward − mean):
  - A1 = 1 − 0.25 = +0.75
  - A2 = 0 − 0.25 = −0.25
  - A3 = 0 − 0.25 = −0.25
  - A4 = 0 − 0.25 = −0.25

### Step 3: Policy update (intuition)
The training objective will push the model so that:
- The probability of candidate **1** (“4”) increases (positive advantage).
- The probability of candidates **2–4** decreases (negative advantage).
- Meanwhile, a KL term keeps the updated model from becoming too different from a reference model.

After training, you’d expect the model to generate “4” more often for that prompt.

---

## If you tell me your context, I can tailor the example
For instance: are you thinking of GRPO as used with (a) a **verifier/reward model**, (b) **rule-based rewards**, or (c) **tool-use / structured outputs**?

## 4) Try your own question

Edit `question` in the previous cell and re-run.